In [6]:
import transformers
from transformers import pipeline

#pip install transformers torch 
#version 5.2.0
#print(transformers.__version__)
# Load the model
# This should be done only once during application startup
# Initializing the model for every request is not recommended due to performance overhead
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)



C:\Users\sreek\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 201/201 [00:00<00:00, 982.12it/s, Materializing param=roberta.encoder.layer.11.output.dense.we
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import nest_asyncio
from typing import List
nest_asyncio.apply()
import io
from fastapi import FastAPI,HTTPException
from fastapi.responses import RedirectResponse
import uvicorn


app = FastAPI(docs_url="/docs")

@app.get("/")
def redirect_to_docs():
    return RedirectResponse(url="http://127.0.0.1:8080/docs")

@app.post("/find_sentiments" ,summary="Find sentiments for list of texts",
    description="This API accepts an array of strings and returns sentiment results.")
def find_sentiments(items: List[str]):
    print("Inside the smentiments process_items")
    return findsentiments(items)

config = uvicorn.Config(app, host="127.0.0.1",port=8080)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [10196]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)


INFO:     127.0.0.1:60212 - "GET / HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:60212 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:60212 - "GET /openapi.json HTTP/1.1" 200 OK
Inside the smentiments process_items
INFO:     127.0.0.1:52163 - "POST /find_sentiments HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [10196]


In [4]:
# Determine sentiment category based on model confidence scores
def convert_sentiment(label, confidence):
    LABEL_MAP = {
    "LABEL_0": "Negative",
    "LABEL_1": "Neutral",
    "LABEL_2": "Positive"
    }
    
    sentiment = LABEL_MAP.get(label, "Unknown")
    weight = round(confidence * 100, 2)

    if confidence < 0.60:
        return {
            "sentiment": sentiment,
            "weightage": weight,
            "strength": "Low (Mixed)"
        }
    elif confidence < 0.75:
        return {
            "sentiment": sentiment,
            "weightage": weight,
            "strength": "Medium"
        }
    else:
        return {
            "sentiment": sentiment,
            "weightage": weight,
            "strength": "High"
        }

In [3]:
def findsentiments(texts):
    results_list = []
    results = sentiment_pipeline(texts)
    #print("The result is " , results)
    # Display results
    for text, result in zip(texts, results):
         results_list.append(
         convert_sentiment(result["label"], result["score"])
        )
    return(results_list)